# A. CNN

## 1. Convolution 2D (From Scratch)

In [ ]:
class Conv2d(nn.Module):
    def __init__(self, in_channels, out_channels, kernel_size, stride=1, padding=0, bias=True):
        super().__init__()
        
        if isinstance(kernel_size, int):
            kernel_size = (kernel_size, kernel_size)
        if isinstance(stride, int):
            stride = (stride, stride)
        if isinstance(padding, int):
            padding = (padding, padding)
        
        self.in_channels = in_channels
        self.out_channels = out_channels
        self.kernel_size = kernel_size
        self.stride = stride
        self.padding = padding
        
        # Initialize weights and bias
        self.weight = nn.Parameter(torch.randn(out_channels, in_channels, *kernel_size))
        if bias:
            self.bias = nn.Parameter(torch.zeros(out_channels))
        else:
            self.register_parameter('bias', None)
        
        # Xavier initialization for weights
        nn.init.xavier_uniform_(self.weight)
    
    def forward(self, x):
        # x: [batch_size, in_channels, height, width]
        batch_size, in_channels, h, w = x.shape
        
        # Apply padding
        if self.padding[0] > 0 or self.padding[1] > 0:
            x = torch.nn.functional.pad(x, (self.padding[1], self.padding[1], self.padding[0], self.padding[0]))
        
        # Calculate output dimensions
        out_h = (x.shape[2] - self.kernel_size[0]) // self.stride[0] + 1
        out_w = (x.shape[3] - self.kernel_size[1]) // self.stride[1] + 1
        
        # Extract patches for convolution using unfold
        patches = torch.nn.functional.unfold(x, kernel_size=self.kernel_size, stride=self.stride)
        # patches shape: [batch_size, in_channels * kernel_h * kernel_w, out_h * out_w]
        
        # Reshape weight for matrix multiplication
        weight_reshaped = self.weight.view(self.out_channels, -1)  # [out_channels, in_channels * kernel_h * kernel_w]
        
        # Apply convolution via matrix multiplication
        out = torch.matmul(weight_reshaped, patches)  # [batch_size, out_channels, out_h * out_w]
        
        # Add bias
        if self.bias is not None:
            out = out + self.bias.view(1, -1, 1)
        
        # Reshape output
        out = out.view(batch_size, self.out_channels, out_h, out_w)
        return out

## 2. Maxpooling (From Scratch)

In [ ]:
class MaxPool2d(nn.Module):
    def __init__(self, kernel_size, stride=None, padding=0):
        super().__init__()
        
        if isinstance(kernel_size, int):
            kernel_size = (kernel_size, kernel_size)
        if isinstance(padding, int):
            padding = (padding, padding)
        
        self.kernel_size = kernel_size
        self.stride = stride if stride is not None else kernel_size
        self.padding = padding
    
    def forward(self, x):
        # x: [batch_size, channels, height, width]
        if isinstance(self.stride, int):
            stride = (self.stride, self.stride)
        else:
            stride = self.stride
        
        # Apply padding
        if self.padding[0] > 0 or self.padding[1] > 0:
            x = torch.nn.functional.pad(x, (self.padding[1], self.padding[1], self.padding[0], self.padding[0]))
        
        batch_size, channels, h, w = x.shape
        
        # Calculate output dimensions
        out_h = (h - self.kernel_size[0]) // stride[0] + 1
        out_w = (w - self.kernel_size[1]) // stride[1] + 1
        
        # Extract patches using unfold
        patches = torch.nn.functional.unfold(x, kernel_size=self.kernel_size, stride=stride)
        # patches shape: [batch_size, channels * kernel_h * kernel_w, out_h * out_w]
        
        patches = patches.view(batch_size, channels, self.kernel_size[0] * self.kernel_size[1], out_h * out_w)
        
        # Apply max pooling
        out, _ = torch.max(patches, dim=2)
        # out shape: [batch_size, channels, out_h * out_w]
        
        # Reshape back to spatial dimensions
        out = out.view(batch_size, channels, out_h, out_w)
        return out

## 3. Linear Layer from scratch

In [ ]:
class Linear(nn.Module):
    def __init__(self, in_features, out_features, bias=True):
        super().__init__()
        self.in_features = in_features
        self.out_features = out_features
        
        # Initialize weights and bias
        self.weight = nn.Parameter(torch.randn(out_features, in_features))
        if bias:
            self.bias = nn.Parameter(torch.zeros(out_features))
        else:
            self.register_parameter('bias', None)
        
        # Xavier initialization for weights
        nn.init.xavier_uniform_(self.weight)
    
    def forward(self, x):
        # x: [batch_size, ..., in_features]
        return torch.nn.functional.linear(x, self.weight, self.bias)

## 4. Flatten and ReLu from Scratch

In [ ]:
class Flatten(nn.Module):
    def __init__(self, start_dim=1, end_dim=-1):
        super().__init__()
        self.start_dim = start_dim
        self.end_dim = end_dim
    
    def forward(self, x):
        return torch.flatten(x, start_dim=self.start_dim, end_dim=self.end_dim)
    
class ReLU(nn.Module):
    def __init__(self, inplace=False):
        super().__init__()
        self.inplace = inplace
    
    def forward(self, x):
        if self.inplace:
            return x.clamp_(min=0)
        else:
            return torch.clamp(x, min=0)

## 5. Entire CNN from scratch

In [ ]:
class CNNFromScratch(nn.Module):
    """
    A complete CNN network built from scratch without using nn.Conv2d, nn.Linear, etc.
    This network is designed for CIFAR-10 or MNIST classification tasks.
    
    Architecture:
    - Input: [batch_size, 3, 32, 32] (CIFAR-10) or [batch_size, 1, 28, 28] (MNIST)
    - Conv Block 1: Conv2d(32 filters) -> ReLU -> MaxPool2d
    - Conv Block 2: Conv2d(64 filters) -> ReLU -> MaxPool2d
    - Conv Block 3: Conv2d(128 filters) -> ReLU -> MaxPool2d
    - Flatten
    - Dense Block: Linear(256) -> ReLU -> Dropout -> Linear(num_classes)
    """
    def __init__(self, in_channels=3, num_classes=10, dropout_rate=0.5):
        super().__init__()
        
        # Convolutional blocks
        self.conv1 = Conv2d(in_channels, 32, kernel_size=3, stride=1, padding=1)
        self.relu1 = ReLU()
        self.pool1 = MaxPool2d(kernel_size=2, stride=2, padding=0)
        
        self.conv2 = Conv2d(32, 64, kernel_size=3, stride=1, padding=1)
        self.relu2 = ReLU()
        self.pool2 = MaxPool2d(kernel_size=2, stride=2, padding=0)
        
        self.conv3 = Conv2d(64, 128, kernel_size=3, stride=1, padding=1)
        self.relu3 = ReLU()
        self.pool3 = MaxPool2d(kernel_size=2, stride=2, padding=0)
        
        # Flatten layer
        self.flatten = Flatten(start_dim=1)
        
        # Fully connected layers
        # For CIFAR-10/MNIST: after 3 pooling layers, spatial size becomes 4x4 or 3x3
        # 128 channels * 4 * 4 = 2048 or 128 * 3 * 3 = 1152
        self.fc1 = Linear(128 * 4 * 4, 256)
        self.relu_fc1 = ReLU()
        self.dropout = Dropout(p=dropout_rate)
        
        self.fc2 = Linear(256, num_classes)
    
    def forward(self, x):
        # Conv Block 1
        x = self.conv1(x)
        x = self.relu1(x)
        x = self.pool1(x)
        
        # Conv Block 2
        x = self.conv2(x)
        x = self.relu2(x)
        x = self.pool2(x)
        
        # Conv Block 3
        x = self.conv3(x)
        x = self.relu3(x)
        x = self.pool3(x)
        
        # Flatten
        x = self.flatten(x)
        
        # Fully connected layers
        x = self.fc1(x)
        x = self.relu_fc1(x)
        x = self.dropout(x)
        x = self.fc2(x)
        
        return x


# Example usage:
# model = CNNFromScratch(in_channels=3, num_classes=10, dropout_rate=0.5)
# x = torch.randn(32, 3, 32, 32)  # Batch of 32 CIFAR-10 images
# output = model(x)  # [32, 10]
# print(f"Output shape: {output.shape}")

# B. Transformers

## 1. Positional Encoding (Trigo - Sine Cosine)

In [ ]:
def generate_positional_encoding(seq_length, dim):
    if dim % 2 != 0:
        raise ValueError("dim must be even for sinusoidal positional encoding")

    pe = torch.zeros(seq_length, dim)
    position = torch.arange(0, seq_length, dtype=torch.float).unsqueeze(1)
    div_term = torch.exp(torch.arange(0, dim, 2).float() * (-torch.log(torch.tensor(10000.0)) / dim))
    pe[:, 0::2] = torch.sin(position * div_term)
    pe[:, 1::2] = torch.cos(position * div_term)
    return pe
# Version 2
def forward(self):
    even_i = torch.arange(0, self.d_model, 2).float()
    denominator = torch.pow(10000, even_i/self.d_model)
    position = torch.arange(self.max_sequence_length).float().reshape(self.max_sequence_length,1)
    even_PE = torch.sin(position/denominator)
    odd_PE = torch.cos(position/denominator)
    stacked = torch.stack((even_PE, odd_PE), dim=2)
    PE = torch.flatten(stacked, start_dim=1, end_dim=2)
    return PE

## 2. Transformer Block

Includes MHA, Layer Normalisation, Residual Connection (dropout), MLP

Flow: LayerNorm >> MHA >> Residual Connection>> LayerNorm >> MLP

In [ ]:
class TransformerBlock(nn.Module):
    def __init__(self, d, num_heads, dropout):
        super().__init__()
        self.LN_MHA = nn.LayerNorm(d)
        self.LN_MLP = nn.LayerNorm(d)
        self.MHA = MultipleAttentionHead(d, num_heads, dropout)
        self.MLP = nn.Sequential(nn.Linear(d,4*d), nn.ReLU(), nn.Dropout(dropout), nn.Linear(4*d,d))         
    def forward(self, H): # size=[batch_size, seq_length, d]
        # Multiple Attention Heads w/ layer normalization (LN), residual connection (RC)
        H = H + self.MHA(self.LN_MHA(H)) # size=[batch_size, seq_length, d]
        # MLP w/ layer normalization (LN), residual connection (RC)
        H = H + self.MLP(self.LN_MLP(H)) # size=[batch_size, seq_length, d]
        return H # size=[batch_size, seq_length, d]

### 2.1. Multi Head Attention

In [ ]:
class MultipleAttentionHead(nn.Module):
    def __init__(self, d, num_heads, dropout):
        super().__init__()
        d_head = d // num_heads # dim_head = d // num_heads, usually dimension per head is 64
        assert d == d_head * num_heads # check divisibility
        self.MHA = nn.ModuleList([ AttentionHead(d, d_head, dropout) for _ in range(num_heads) ])
        self.WO = nn.Linear(d, d) # combination layer
        self.dropout = nn.Dropout(dropout)
    def forward(self, H): # size(H)=[batch_size, seq_length, d]
        batch_size = H.size(0); seq_length = H.size(1)
        H_heads = []
        for HA_layer in self.MHA:
            H_heads.append(HA_layer(H)) # size=[batch_size, seq_length, d_head]
        H_heads = torch.cat(H_heads, dim=2) # size=[batch_size, seq_length, d]            
        H_heads = self.dropout(H_heads) # dropout attention activations
        H = self.WO(H_heads) # size=[batch_size, seq_length, d]
        return H

In [ ]:
class AttentionHead(nn.Module):
    def __init__(self, d, d_head, dropout):
        super().__init__()
        self.LN_MHA = nn.LayerNorm(d_head)
        self.LN_MLP = nn.LayerNorm(d_head)
        self.query = nn.Linear(d, d_head, bias=False) # query embedding layer
        self.key = nn.Linear(d, d_head, bias=False) # key embedding layer
        self.value = nn.Linear(d, d_head) # value embedding layer
        self.dropout = nn.Dropout(dropout)
    def forward(self, H): # size(H)=[batch_size, seq_length, d]
        batch_size = H.size(0)
        seq_len = H.size(1) # batch_len     
        H_in = H # Add residual connection (RC)

        d_k = Q.size(2) # dimension of key (or query) vectors 
        # Compute a single attention head H = Softmax( QK^T / d^0.5 ) V
        Q = self.query(H) # size=[batch_size, batch_length, d]        
        K = self.key(H) # size=[batch_size, batch_length, d]
        V = self.value(H) # size=[batch_size, batch_length, d]
           
        attention_score = Q @ K.transpose(-2,-1) * torch.sqrt(torch.tensor(d_k, dtype=torch.float32)) # QK^T/sqrt(d), (B,L,d) @ (B,d,L) => (B,L,L), size=[batch_size, batch_length, batch_length)
        # Generating the mask to prevent using next tokens for prediction
        mask = torch.tril(torch.ones(seq_len , seq_len)).long().to(device) # mask to use previous tokens only : { token(<=t) }, size=[batch_len,batch_len]
        # Apply Masking
        attention_score = attention_score.masked_fill(mask==0, value=float('-inf')) # softmax(-inf)=0 prevents using next tokens for prediction, size=(batch_size, batch_len, batch_len)
        attention_score = torch.softmax(attention_score, dim=-1) # sum weights = 1, size=[batch_size, batch_length, batch_len)
        # Apply dropout to attention scores for regularization 
        attention_score = self.dropout(attention_score) # dropout attention scores
        H_HA = attention_score @ V # softmax( QK^T / sqrt(d) ) V, (B,L,L) @ (B,L,d) => (B,L,d), size=[batch_size, batch_length, d)
        return H_HA # return prediction scores for next token

### 2.3. Layer Normalisation (From Scratch)

In [ ]:
class LayerNorm(nn.Module):
    def __init__(self, normalized_shape, eps=1e-5, elementwise_affine=True):
        super().__init__()

        if isinstance(normalized_shape, int):
            normalized_shape = (normalized_shape,)
        elif isinstance(normalized_shape, list):
            normalized_shape = tuple(normalized_shape)

        self.normalized_shape = normalized_shape
        self.eps = eps
        self.elementwise_affine = elementwise_affine

        if self.elementwise_affine:
            self.weight = nn.Parameter(torch.ones(self.normalized_shape))
            self.bias = nn.Parameter(torch.zeros(self.normalized_shape))
        else:
            self.register_parameter("weight", None)
            self.register_parameter("bias", None)

    def forward(self, x):
        # Normalize over the last len(normalized_shape) dimensions.
        dims = tuple(range(-len(self.normalized_shape), 0))
        mean = x.mean(dim=dims, keepdim=True)
        var = x.var(dim=dims, keepdim=True, unbiased=False)
        y = (x - mean) / torch.sqrt(var + self.eps)

        if self.elementwise_affine:
            y = y * self.weight + self.bias
        return y


# Route later nn.LayerNorm(...) calls to this from-scratch implementation.
nn.LayerNorm = LayerNorm

### 2.4. Dropout from scratch

In [ ]:
class Dropout(nn.Module):
    def __init__(self, p=0.5, inplace=False):
        super().__init__()

        if not 0.0 <= p <= 1.0:
            raise ValueError(f"dropout probability has to be in [0, 1], but got {p}")
        if inplace:
            raise NotImplementedError("inplace=True is not implemented in this from-scratch version")

        self.p = p
        self.inplace = inplace

    def forward(self, x):
        # During evaluation, dropout is disabled.
        if (not self.training) or self.p == 0.0:
            return x

        # If p == 1, all activations are dropped.
        if self.p == 1.0:
            return torch.zeros_like(x)

        # Inverted dropout: scale by 1 / (1 - p) so E[output] = input.
        keep_prob = 1.0 - self.p
        mask = (torch.rand_like(x) < keep_prob).to(x.dtype) / keep_prob
        return x * mask


# Route later nn.Dropout(...) calls to this from-scratch implementation.
nn.Dropout = Dropout